# 00 — Run all notebooks one after another

Runs `04_yolo → 02_faster_rcnn → 03_retinanet → 05_rtdetr → 06_final_pipeline` in order. Each executed
notebook, with all outputs, is saved to `MyDrive/vdet/executed/` (Colab) or `~/vdet_work/executed/` (laptop).

* If the session stops (disconnect, GPU limit): open this notebook again and **Run all**. Finished
  notebooks are skipped, the interrupted one continues from its last finished epoch.
* If one notebook fails, the error is reported and the next notebook still runs.

**Colab:** Runtime → Change runtime type → T4 GPU. Kaggle token as a Colab secret: key icon on the
left → Add new secret → name `KAGGLE_API_TOKEN` → paste the token → switch on *Notebook access*.

In [ ]:
# ---- Settings ----
ORDER = ["04_yolo", "02_faster_rcnn", "03_retinanet", "05_rtdetr", "06_final_pipeline"]
SMOKE = False   # True: very short runs, only to check that everything works (~1 h for all)
FAST = True     # see the notebooks: every 20th train frame and fewer epochs
REPO = "https://github.com/LidaSokha03/AI_HW2.git"

In [ ]:
import os
import sys
import json
import time
import subprocess

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    from google.colab import drive, userdata
    drive.mount("/content/drive")
    try:  # Kaggle token from the Colab secrets -> file, so that the executed notebooks can read it
        token = userdata.get("KAGGLE_API_TOKEN")
    except Exception:
        token = None
    if token:
        os.makedirs(os.path.expanduser("~/.kaggle"), exist_ok=True)
        with open(os.path.expanduser("~/.kaggle/access_token"), "w") as f:
            f.write(token.strip())
        os.chmod(os.path.expanduser("~/.kaggle/access_token"), 0o600)
        print("Kaggle token: found in Colab secrets")
    else:
        print("Kaggle token: not in Colab secrets - kaggle.json in My Drive will be used if present")
    REPO_DIR, WORK = "/content/AI_HW2", "/content/drive/MyDrive/vdet"
    if os.path.exists(REPO_DIR):
        subprocess.run(["git", "-C", REPO_DIR, "pull", "-q"], check=True)
    else:
        subprocess.run(["git", "clone", "-q", "--depth", "1", REPO, REPO_DIR], check=True)
else:
    REPO_DIR = os.environ.get("VDET_REPO", os.path.abspath(".."))
    WORK = os.environ.get("VDET_WORK", os.path.expanduser("~/vdet_work"))

if SMOKE:  # test runs go to a separate folder
    WORK = f"{WORK}/smoke"
gpu = subprocess.run("nvidia-smi -L", shell=True, capture_output=True, text=True).stdout.strip()
print("GPU:", gpu or "NOT FOUND - Runtime -> Change runtime type -> T4 GPU")
OUT = f"{WORK}/executed"
os.makedirs(OUT, exist_ok=True)
print("code:", REPO_DIR, "| executed notebooks ->", OUT)

In [ ]:
def first_error(path):
    """First error in an executed notebook, or None."""
    nb = json.load(open(path))
    for cell in nb["cells"]:
        for o in cell.get("outputs", []):
            if o.get("output_type") == "error":
                return f"{o['ename']}: {o['evalue'][:300]}"
    return None


status = {}
for name in ORDER:
    done_flag = f"{OUT}/{name}.done"
    if os.path.exists(done_flag) and not SMOKE:
        status[name] = "done earlier"
        print(f"{name}: done earlier - skipped")
        continue
    src = open(f"{REPO_DIR}/notebooks/{name}.ipynb").read()
    src = src.replace("SMOKE = False", f"SMOKE = {SMOKE}").replace("FAST = True", f"FAST = {FAST}")
    tmp = f"{REPO_DIR}/notebooks/_run_{name}.ipynb"   # executed from notebooks/, so ../src is found
    with open(tmp, "w") as f:
        f.write(src)
    t0 = time.time()
    print(f"{time.strftime('%H:%M')}  {name}: running...", flush=True)
    r = subprocess.run([sys.executable, "-m", "jupyter", "nbconvert", "--to", "notebook", "--execute",
                        "--allow-errors", "--ExecutePreprocessor.timeout=-1", tmp,
                        "--output-dir", OUT, "--output", name], capture_output=True, text=True)
    minutes = (time.time() - t0) / 60
    out_nb = f"{OUT}/{name}.ipynb"
    err = r.stderr[-500:] if r.returncode != 0 or not os.path.exists(out_nb) else first_error(out_nb)
    if err:
        status[name] = f"FAILED after {minutes:.0f} min - {err}"
    else:
        status[name] = f"ok ({minutes:.0f} min)"
        if not SMOKE:
            open(done_flag, "w").write(time.strftime("%Y-%m-%d %H:%M"))
    print(f"{time.strftime('%H:%M')}  {name}: {status[name]}", flush=True)

## Summary

In [ ]:
import glob
import pandas as pd

print("\n".join(f"{k:20s} {v}" for k, v in status.items()))
print("\nexecuted notebooks:", sorted(os.path.basename(p) for p in glob.glob(f"{OUT}/*.ipynb")))
print("submissions:", sorted(os.path.basename(p) for p in glob.glob(f"{WORK}/submissions/*.csv")))
files = glob.glob(f"{WORK}/results/*.csv")
if files:
    res = pd.concat([pd.read_csv(f) for f in files if "summary" not in f], ignore_index=True)
    cols = [c for c in ["notebook", "run", "mAP", "AP50", "AP_small", "FPS"] if c in res.columns]
    display(res[cols].sort_values(["notebook", "mAP"]))